# SFT-разогрев T-lite с LoRA

Решение 03.11.2026 (`docs/decisions.md`, раздел 9). Примеры собираются в токены функцией среды, как роллауты RL;
loss — только по репликам модели; `transformers.Trainer` + LoRA (peft), логиты — только в позициях с меткой.
Отложено 9 тем SFT: по loss на них выбирается эпоха. Результат — `runs/sft/<дата>/` (адаптер, журнал, разбиение).

Перед запуском: `pip install --user -e ".[models,training]"`; веса T-lite должны быть доступны (Hugging Face или
локальный кеш в `HF_HUB_CACHE`).

In [1]:
import json, os, sys, time
from pathlib import Path
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # перед запуском проверить nvidia-smi
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))  # если пакет не установлен через pip install -e .
import torch
from presentation_rl.environment import build_prompt
from presentation_rl.model_loader import load_model
from presentation_rl.prompts import system_prompt
from presentation_rl.training import sft

CONFIG = "sft_tlite_7ep.yaml"  # первый запуск (3 эпохи) — sft_tlite.yaml
config = sft.SFTConfig.load(ROOT / "configs" / "experiments" / CONFIG)
OUT = ROOT / "runs" / "sft" / time.strftime("%Y%m%d_%H%M")
OUT.mkdir(parents=True, exist_ok=True)
print(config, OUT, torch.cuda.get_device_name(0), sep="\n")

mkdir -p failed for path /home/jovyan/.config/matplotlib: [Errno 20] Not a directory: '/home/jovyan/.config/matplotlib'
Matplotlib created a temporary cache directory at /tmp/matplotlib-i6unb9j7 because there was an issue with the default path (/home/jovyan/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


SFTConfig(model_id='t-tech/T-lite-it-2.1', revision='main', lora_rank=32, lora_alpha=32, learning_rate=0.0002, epochs=3, grad_accum=8, warmup_ratio=0.03, eval_tasks=9, seed=0, max_tokens=16000)
/home/jovyan/Diplom/runs/sft/20261107_1525
Tesla V100-SXM3-32GB


In [2]:
start = time.time()
model, tokenizer = load_model(config.model_id, config.revision, dtype="float16", device_map={"": 0})
print(f"загрузка {time.time() - start:.0f} с, память {torch.cuda.memory_allocated() / 2**30:.1f} ГБ")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

загрузка 687 с, память 15.3 ГБ


## Примеры токенами

Эпизод каждого примера собирается так же, как роллаут: начало — шаблон чата (системный промпт, инструменты, запрос),
дальше реплики модели и выдача инструментов в порядке среды. Слишком длинные эпизоды не обрезаются (оборвался бы
ответ), а отбрасываются — их список печатается.

In [3]:
def prompt(request):
    return build_prompt(tokenizer, system_prompt(request), request)


samples = sft.build_samples(ROOT, tokenizer, prompt)
samples, dropped = sft.drop_long(samples, config.max_tokens)
print("отброшены как слишком длинные:", [(s.id, s.kind, len(s.ids)) for s in dropped])
train_samples, eval_samples = sft.split(samples, config.eval_tasks, config.seed)
for name, part in (("обучение", train_samples), ("отложенные", eval_samples)):
    print(name, sft.lengths(part))
(OUT / "split.json").write_text(json.dumps({"eval_topics": sorted({s.id for s in eval_samples}),
                                            "dropped": [[s.id, s.kind, len(s.ids)] for s in dropped]},
                                           ensure_ascii=False, indent=1), encoding="utf-8")

отброшены как слишком длинные: []
обучение {'examples': 121, 'max': 12085, 'median': 4614, 'total': 677424, 'target_total': 240469}
отложенные {'examples': 16, 'max': 10874, 'median': 5296, 'total': 103811, 'target_total': 33387}


129

## Обучение

In [4]:
start = time.time()
torch.cuda.reset_peak_memory_stats()
model = sft.train(config, train_samples, eval_samples, OUT, model=model, tokenizer=tokenizer)
print(f"обучение {(time.time() - start) / 60:.0f} мин, пик памяти {torch.cuda.max_memory_allocated() / 2**30:.1f} ГБ")
log = json.loads((OUT / "log.json").read_text(encoding="utf-8"))
for row in log:
    if "eval_loss" in row:
        print(f"эпоха {row['epoch']:.0f}: loss на отложенных темах {row['eval_loss']:.4f}")

trainable params: 87,293,952 || all params: 8,276,005,888 || trainable%: 1.0548


[W1107 15:37:20.411082265 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 2227175424 bytes (free: 1634729984, total: 34072559616).


Epoch,Training Loss,Validation Loss
1,1.601600,1.467788
2,1.259845,1.436484
3,1.454655,1.430011


[W1107 15:45:57.755416536 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 2323644416 bytes (free: 1364197376, total: 34072559616).


обучение 40 мин, пик памяти 24.8 ГБ
эпоха 1: loss на отложенных темах 1.4678
эпоха 2: loss на отложенных темах 1.4365
эпоха 3: loss на отложенных темах 1.4300


## Быстрая проверка

Один ответ на отложенную тему без поиска (полная проверка на validation — EXP-015, отдельная тетрадка).

In [5]:
from presentation_rl.dsl import parse
from presentation_rl.environment import run_episode
from presentation_rl.model_loader import hf_generator

model.eval()
model.config.use_cache = True
record = next(json.loads(l) for l in (ROOT / sft.PLAIN).read_text(encoding="utf-8").splitlines()
              if json.loads(l)["id"] in {s.id for s in eval_samples})
from presentation_rl.tasks import load_tasks
request = sft.request_for(record, {t.id: t for t in load_tasks(ROOT / "data" / "tasks" / "tasks.json", split="sft")})
episode = run_episode(prompt(request), tokenizer, hf_generator(model, tokenizer, temperature=0.9), None, max_new_tokens=6000)
deck = parse(episode.answer)
print(f"{record['id']}: слайдов {len(deck.slides)}, корректных строк {deck.parse_rate:.0%}, конец — {episode.finish}")
print(episode.answer[:3000])

t006: слайдов 20, корректных строк 93%, конец — stop
#PLAN: хронология Древнего Востока → хронология Древнего Египта → хронология Месопотамии → хронология Древнего Китая → хронология Индийского субконтинента → хронология Древнего Ирана → периодизация Древнего Востока → периодизация Древнего Египта → периодизация Древнего Ближнего Востока → периодизация Древнего Китая → периодизация Древней Индии → особенности развития Древнего Востока → особенности Древнего Египта → особенности Древнего Ближнего Востока → особенности Древнего Китая → особенности Древней Индии → особенности Древнего Ирана → заключение
#SLIDE
#TITLE: История Древнего Востока: хронология, периодизация, особенности развития
#TEXT: Дату первого упоминания названия региона в письменных источниках, исторические периоды, особенности цивилизаций и причины сходства и различий обществ на обширном пространстве.
#SLIDE
#TIMELINE: ок. 3150 — объединение Верхнего и Нижнего Египта; ок. 2000 до н. э. — начало правления гиксосов в Египт